In [7]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Literal , Annotated
from dotenv import load_dotenv
from pydantic import BaseModel , Field  
from langchain_core.messages import HumanMessage , SystemMessage , BaseMessage
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [4]:
class JokeState(TypedDict):
    topic:str 
    joke:str
    explanation:str

In [ ]:
def generate_joke(state:JokeState):
    prompt=f"generate a joke on topic{state['topic']}"
    answer = llm.invoke(prompt).content
    return {'joke':answer}

In [ ]:
def generate_explanation(state:JokeState):
    prompt=f"generate a explanation for the joke {state['joke']}"
    answer = llm.invoke(prompt).content
    return {'explanation':answer}

In [8]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke' , generate_joke)
graph.add_node('generate_explanation' ,generate_explanation)

graph.add_edge(START , 'generate_joke')
graph.add_edge('generate_joke' , 'generate_explanation')
graph.add_edge('generate_explanation' ,END )


checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)
 


In [9]:
config1= {'configurable':{'thread_id':'1'}}

workflow.invoke({'topic':'pizza'} , config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': AIMessage(content=[{'type': 'text', 'text': 'What did the pizza delivery guy say to the comedian?\n\n**"You can\'t top my delivery!"**', 'extras': {'signature': 'Es8eCsweARFNMg/lLnYiGxTBUm0MIemB/Qwtr/osrbdfsqwOFYGBErG/nCj3cz/fFYJe6eNO5T4MtXDPOsgXfWnAQwd9e0szuYaz/0SNtv5THGn4Hd9e5fHt39K1TZe2THb2tZOLHNH8US43LW4CeMklfpl3M9NoZHrEQoVdOej7pt61L/Ooi0tgjFsB4a791p7ey1yhx+IV4y53ZZDnFKJ0arzQ2OiOqYMhq+3Vc+RxxvqSJhj3O72qvoMj5/haKhWXBnIJiaq1yohbqreaNgiCAygntziFzz9UZ4vpatAATJ/Cu0FH3S9Owef6+JUQT0yBgmDvUCesbk1X1BjCTerYoJXwNzISqU3a4JJbv25hIWRRr6ARsMmyWhgUVUuiFR/ltRmTlPAJQop+qpyccIu3u9QbLV5DUHqewx/ua9/OiUsJy4/4+88J6f/P/WHQfQHZrbnQ71cVYD5BiNakwEw3W2aL+ANH6HTmupF6YVJAoXHq5u+qAmPBYAFhEBvLulBEtnezRkvJVQ04hqinVEa5hUwzlMjkezZHszQoqgveF7ccT7I+H+YsDfYu9x8DSbAKXfD4w6e3dN78HwnPhYOzfz1ycGSxP8XB4HhGgzsJEFxiXE0t3JmzrDSXW93YRrw3JGbhu3Tu9U2P/TASLTulKD4DQ2V4nmi0Yds9L3gvJUh4fqidyH0+uP7e7JtglmDiDEj+nIr/upI1CZrlkyHEu4IxyJgHsU21ceh0pCzMWEWQFBOCHicjmyu0S042g5v55KOgHj2Xz6UNHxras1Ogj2dHm1LES8jZIhvZXj